# Stage 2 Notebook 36 - Exp2EE Cosine LR fix for the late-training collapse

**Why this exists.** NB34 (Exp2CC) revealed a classic deep-learning training pathology, not an architectural bottleneck:

| epoch | matched_iou | val_lane_line_iou_loss | decoded_f1 | oracle_f1 |
|---|---:|---:|---:|---:|
| 18 | **0.157** | 0.843 | 0.040 | **0.100** ← peak |
| 23 | **0.050** | **0.950** | 0.009 | 0.011 |
| 28 | 0.048 | 0.952 | 0.019 | 0.022 |

Geometry exploded between epoch 18 and 23 while cls stayed stable. **This is constant-LR Adam stepping out of a converged minimum**, particularly easy on the non-smooth LineIoU loss whose gradient is discontinuous at exact-overlap and zero-overlap boundaries. Without LR decay, the optimizer keeps full step size near convergence and eventually steps OUT.

Every published lane detector uses cosine LR with linear warmup (CLRKDNet, CLRNet, RMT-PPAD). We never had it. NB36 introduces it.

Diff vs Exp2Z (NB31, exp26):
- New `train.lr_scheduler: {kind: cosine, warmup_epochs: 2, warmup_start_lr_factor: 0.1, min_lr_factor: 0.05}`.
- `train.end_epoch: 15 -> 20` (cosine decay needs more epochs to amortize the 2-epoch warmup; 20 gives a clean 18-epoch decay window).
- All other settings identical.

Implementation: `stage2/scripts/train_joint_model_experiment.py` now has a cosine LambdaLR scheduler with linear warmup. Logged per-epoch LR shows decay from 0.0002 -> ~0.00001.

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After smoke + debug pass, change to `False` for the 20-epoch short run.
3. Output mirrored to notebook cell, Colab runtime log, Drive log file.
4. Do not rerun NB00.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp31_rmt_gca_mask_uncertainty_cosine_lr_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp31_rmt_gca_mask_uncertainty_cosine_lr_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp31_rmt_gca_mask_uncertainty_cosine_lr_joint_smoke.log
OK exp31_rmt_gca_mask_uncertainty_cosine_lr_joint.yaml
  lane_shape=(1, 12, 72, 2) det_shape=(1, 4, 4)
  lane_loss=3.9802 det_loss=3.2843 grad_cos=-0.2042 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.5003716349601746, 'gate/lane_mean': 0.5003561973571777, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp31_rmt_gca_mask_uncertainty_cosine_lr_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short20'
    EPOCHS = 20
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp31_rmt_gca_mask_uncertainty_cosine_lr_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp31_rmt_gca_mask_uncertainty_cosine_lr_joint_short20 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp31_rmt_gca_mask_uncertainty_cosine_lr_joint_short20.tar --epochs 20 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp31_rmt_gca_mask_uncertainty_cosine_lr_joint_short20.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp31_rmt_gca_mask_uncertainty_cosine_lr_joint_short20_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp31_rmt_gca_mask_uncertainty_cosine_lr_joint.yam

0

## What to watch in Exp2EE training

Pass criteria at epoch 20:
- **NO COLLAPSE**: `val/matched_line_iou` never drops below 0.10 in any epoch. NB34 collapsed to 0.050 at ep23.
- **`val/matched_line_iou >= 0.16`** sustained, ideally peaks higher than Exp2Z's 0.162.
- **`val/lane/decoded_f1 >= 0.05`**: stable training should beat Exp2Z's 0.044.
- **`val/lane/decoded_oracle_f1 >= 0.10`**.
- **Per-epoch LR log line** shows decay: ep1 ~0.00002 (warmup start), ep2-3 ~0.00020, then cosine decay toward ~0.00001 at ep20.
- **train_total decreases monotonically**, no late-training divergence.

Failure signals:
- decoded_f1 plateaus exactly at 0.044: cosine LR fixed the collapse but couldn't unlock further gains. The architecture's capacity ceiling is the real impasse. Pivot to Exp2FF (KD) for cls breakthrough or to bigger backbone.
- Geometry collapses despite cosine LR: there's a deeper instability. Try `min_lr_factor: 0.0` or smaller `lr0`.